## Generate the $C_\ell$ with chosen mass, injection $\xi$, and $\tau$ value

Here we compute the $C_\ell$ spectra in the case of a certain $\tau$ value and with the injection $\xi$. 

The spectra are computed using pyrefCLASS, a wrapper for birefCLASS, a modified version of CLASS including cosmic birefringence. 

In [3]:
# Setup and imports

import sys, os
import numpy as np
import tqdm
from matplotlib.pyplot import *
from scipy.integrate import solve_ivp
import importlib

dir_pyrefclass = '//Users/salvatoresamuelesirletti/pyrefCLASS'  # directory of pyrefCLASS
sys.path.append(dir_pyrefclass)
import pyrefCLASS as rot
importlib.reload(rot)

import local

Neta = 10000       # number of time steps for output phi(eta)
logm = -30.5        # <-- set here the value of log10(m_phi)

beta0_deg = 0.3                     # for phi/beta plots
beta0_rad = 0.3 * np.pi / 180       # to be passed to CLASS in create_inifile

n, tf = 1, 100       # parameters for the ALP potential

z_start = 60          # <-- start of the energy injection window
z_end = 50            # <-- end of the energy injection window
bphi0_ini = 1.0        # normalized phi_ini

eps = 1e-8    # numerical regularization parameter for 1/phi'
ow = True     # overwrite existing files

Asplanck = 2.1e-9
tau_planck = 0.054
const_planck = 2.1e-9 * np.exp(-2 * tau_planck)

outroot = '/Users/salvatoresamuelesirletti/Desktop/Work/Biref_iso_inject/output/'
os.makedirs(outroot, exist_ok=True)

In [4]:
# Cell 2 - Compute spectrum for given xi and tau


x = 0.0113  
t_single = 0.08986

base_tag = 'inject_tau' + str(t_single) + '_logm' + str(logm) + '_xi' + str(x) + '_N' + str(Neta)
ofile = outroot + base_tag + '_cl.dat'
#npy_file = outroot + base_tag + '_cl.npy'

if os.path.exists(ofile) and not ow:
    print(ofile + ' exist, skipping computation.', end='\n', flush=True)
else:
    # update primordial amplitude consistently with tau
    ln10As = np.log(1e10 * const_planck * np.exp(2 * t_single))
    dynamics_file = rot.dir_phi + base_tag

    if ow and os.path.exists(dynamics_file):
        os.remove(dynamics_file)

    # export the axion field dynamics phi(a) by solving the equation of motion
    rot.export_axion_dynamics_a(Neta, local.EoM_phi, [logm, tf, n, x, eps, z_start, z_end, bphi0_ini], rot.dir_phi + base_tag)
    
    # build the CLASS .ini file for this configuration
    rot.create_inifile(
        beta0_rad, base_tag, Neta, base_tag, default_ini_file='biref_default_inject.ini', tau_reio=t_single, ln10As=ln10As,
        outroot=outroot, phi_dynamics_dir=rot.dir_phi
    )
    
    # run birefCLASS
    os.system(
        rot.path_to_birefCLASS + '/class ' + rot.dir_ini + base_tag + '.ini ' + rot.path_to_birefCLASS + '/cl_ref_biref.pre > test.txt'
    )
    
    # clean up temporary CLASS output files, keep only the _cl.dat spectrum
    for tag in ['cls.dat', 'clt.dat', 'parameters.ini', 'primordial_Pk.dat', 'thermodynamics.dat', 'unused_parameters', 'background.dat']:
        os.system('rm -rf ' + outroot + base_tag + '_' + tag)
    os.system('rm -rf ' + rot.dir_ini + base_tag + '.ini')

# load the computed spectrum and save it as a numpy array
#lcl = np.loadtxt(ofile, unpack=True)
#np.save('best_MCMC_spectrum.npy', lcl)

#print("Computation finished for xi = 0.0113, tau = 0.09. Saved to best_MCMC_spectrum.npy")